# 02 -- Silver Layer Cleaning
**Project:** Airbnb Market Analysis -- New York City  
**Layer:** Silver (cleaned, conformed)  
**Source:** workspace.default.bronze_airbnb_listings  
**Author:** B. Sarang  

Reads the Bronze Delta table, applies data quality rules, casts types correctly, drops unusable records, and writes a clean Silver table ready for analysis.

## 1. Configuration

In [0]:
BRONZE_TABLE = "workspace.default.bronze_airbnb_listings"
SILVER_TABLE = "workspace.default.silver_airbnb_listings"

## 2. Read Bronze table

In [0]:
df = spark.table(BRONZE_TABLE)
print(f"Row count: {df.count()}")
print(f"Columns: {len(df.columns)}")
display(df.limit(5))

## 3. Null audit

In [0]:
from pyspark.sql.functions import col, count, when, round as spark_round

total = df.count()

null_audit = df.select([
    (count(when(col(c).isNull() | (col(c).cast('string') == ''), c)) / total * 100)
    .alias(c)
    for c in df.columns
])

# Transpose for readability
import pandas as pd
null_pd = null_audit.toPandas().T.reset_index()
null_pd.columns = ['column', 'null_pct']
null_pd = null_pd[null_pd['null_pct'] > 0].sort_values('null_pct', ascending=False)
display(null_pd)

## 4. Select and cast relevant columns

In [0]:
from pyspark.sql.functions import regexp_replace, trim
from pyspark.sql.types import DoubleType, IntegerType, LongType

df_selected = df.select(
    col('id').cast(LongType()),
    trim(col('name')).alias('name'),
    trim(col('host_name')).alias('host_name'),
    col('host_id').cast(LongType()),
    col('host_is_superhost'),
    trim(col('neighbourhood_cleansed')).alias('neighbourhood'),
    trim(col('neighbourhood_group_cleansed')).alias('borough'),
    col('latitude').cast(DoubleType()),
    col('longitude').cast(DoubleType()),
    trim(col('room_type')).alias('room_type'),
    regexp_replace(col('price'), '[$,]', '').cast(DoubleType()).alias('price'),
    col('minimum_nights').cast(IntegerType()),
    col('number_of_reviews').cast(IntegerType()),
    col('review_scores_rating').cast(DoubleType()),
    col('reviews_per_month').cast(DoubleType()),
    col('calculated_host_listings_count').cast(IntegerType()),
    col('availability_365').cast(IntegerType()),
    col('last_review'),
    col('_ingested_at')
)

print(f"Columns after selection: {len(df_selected.columns)}")
df_selected.printSchema()

## 5. Apply data quality rules

In [0]:
df_clean = (
    df_selected
    # Drop rows with no price or zero price
    .filter(col('price').isNotNull() & (col('price') > 0))
    # Drop rows with no neighbourhood
    .filter(col('neighbourhood').isNotNull())
    # Drop rows with no room type
    .filter(col('room_type').isNotNull())
    # Cap minimum nights at 365 (outlier filter)
    .filter(col('minimum_nights') <= 365)
    # Cap price at 99th percentile to remove extreme outliers
)

# Remove price outliers above 99th percentile
price_99 = df_clean.approxQuantile('price', [0.99], 0.01)[0]
df_clean = df_clean.filter(col('price') <= price_99)

print(f"Rows after cleaning: {df_clean.count()}")
print(f"Price 99th percentile cap: ${price_99:.2f}")

## 6. Write Silver Delta table

In [0]:
(
    df_clean.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table written: {SILVER_TABLE}")

## 7. Validate Silver table

In [0]:
df_silver = spark.table(SILVER_TABLE)
print(f"Silver row count: {df_silver.count()}")
display(df_silver.limit(10))

## 8. Run OPTIMIZE on Silver table

In [0]:
spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (neighbourhood, room_type)")
print("OPTIMIZE complete")

## 9. Transaction log

In [0]:
spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}").display()